# LangGraph — Conditional Edges and Routing

This is the second "building block" notebook in the series, after `1_LangGraph_Starter.ipynb`.
That notebook covered two ideas — **state** and **reducers** — but every graph in it followed a
fixed path: the same nodes ran in the same order every time.

Real agents need to make a decision and take one path or another. This notebook adds the third
core primitive: **conditional edges** — an edge whose destination is chosen at runtime by a
router function that inspects the current state.

We'll build one graph:

- **Graph C**: a `classify` node asks the LLM to label an incoming question as `"weather"` or
  `"other"`. A conditional edge reads that label and routes to one of two handler nodes.

This is the same pattern `3_Agentic_RAG.ipynb` uses to decide "call a tool" vs. "answer
directly" — isolated here so the routing mechanics are easy to see on their own, with no
retrieval or tools in the way.


GoodMem adaptation of [Chandula Senevirathna’s Agentic_RAG](https://github.com/ChandulaSenevirathna/Agentic_RAG). Original notice: [LICENSE.md](LICENSE.md). Run `uv sync` and configure `.env` first. Diagrams use Mermaid text, so notebook execution needs no external rendering service.

## 1. Imports

Same setup as the starter notebook, plus one addition:

- `typing.Literal` — lets us type-hint the router function's return value as one of a fixed
  set of node names, which is optional but makes the routing intent explicit.


In [ ]:
import os
from typing import Literal

from dotenv import load_dotenv
from IPython.display import Markdown, display
from pydantic import BaseModel

from goodmem_rag.config import chat_model
from langchain_core.messages import HumanMessage

from langgraph.graph import StateGraph, START, END


## 2. Load configuration

The shared configuration reads `.env`. Choose Groq (the upstream default) or Cohere; see `.env.example`. These first two notebooks teach graph mechanics and do not perform retrieval.

In [ ]:
load_dotenv()

## 3. Initialize the LLM

The helper selects the provider and model from `.env` and validates the required key. All graph nodes share this client.

In [ ]:
llm = chat_model()

## Graph C — routing on an LLM decision

### 4. Define the state

Three fields: the incoming `question`, the `label` the classifier assigns to it, and the final
`answer`. No reducers needed here — each field is written exactly once, by exactly one node.


In [ ]:
class RouteState(BaseModel):
    question: str
    label: str = ""
    answer: str = ""


### 5. Define the classify node

Asks the LLM to reduce the question to a single label. The prompt is deliberately strict about
the output format so the router function below has something reliable to parse.


In [ ]:
def classify(state: RouteState):
    prompt = (
        "Classify the following question as exactly one word: either 'weather' or 'other'. "
        "Respond with only that one word.\n\n"
        f"Question: {state.question}"
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    label = "weather" if "weather" in response.content.strip().lower() else "other"
    return {"label": label}


### 6. Define the handler nodes

Two plain nodes, one per branch. `handle_weather` is a stand-in for where a real weather API
call would go; `handle_general` just falls back to asking the LLM directly.


In [ ]:
def handle_weather(state: RouteState):
    return {"answer": "That sounds like a weather question — plug a weather API call in here."}


def handle_general(state: RouteState):
    response = llm.invoke([HumanMessage(content=state.question)])
    return {"answer": response.content}


### 7. Define the router function

This is the new piece. A router is a plain Python function that takes the state and returns the
**name** of the next node as a string, instead of the graph always following one fixed edge.
LangGraph calls it right after `classify` finishes and sends execution wherever it points.


In [ ]:
def route(state: RouteState) -> Literal["handle_weather", "handle_general"]:
    return "handle_weather" if state.label == "weather" else "handle_general"


### 8. Assemble the graph

`add_conditional_edges(source, router, path_map)` wires it up:

- `source` — the node to attach the branch to (`"classify"`)
- `router` — the function that inspects state and returns a node name
- `path_map` — maps the router's possible return values to actual node names (identity here,
  but it's useful when the router returns something shorter than the node name)


In [ ]:
graph_c_builder = StateGraph(RouteState)

graph_c_builder.add_node("classify", classify)
graph_c_builder.add_node("handle_weather", handle_weather)
graph_c_builder.add_node("handle_general", handle_general)

graph_c_builder.add_edge(START, "classify")
graph_c_builder.add_conditional_edges(
    "classify",
    route,
    {"handle_weather": "handle_weather", "handle_general": "handle_general"},
)
graph_c_builder.add_edge("handle_weather", END)
graph_c_builder.add_edge("handle_general", END)

graph_c = graph_c_builder.compile()


### 9. Visualize

The diamond shape at `classify` in the rendered graph is LangGraph's way of marking a
conditional branch point.


In [ ]:
display(Markdown("```mermaid\n" + graph_c.get_graph().draw_mermaid() + "\n```"))


### 10. Run it — a weather question

Expect `label` to come back `"weather"` and `answer` to be the stand-in string from
`handle_weather`, not a real LLM answer — that branch never calls the LLM a second time.


In [ ]:
result = graph_c.invoke({"question": "Will it rain in Colombo tomorrow?"})
print(result["label"])
print(result["answer"])


### 11. Run it — a general question

This one should route to `handle_general` and come back with a real LLM-generated answer.


In [ ]:
result = graph_c.invoke({"question": "What's the capital of France?"})
print(result["label"])
print(result["answer"])


## Next steps

With this notebook, all three core LangGraph primitives are covered: **state**, **reducers**,
and **conditional edges**. `3_Agentic_RAG.ipynb` combines conditional edges with tool-calling so
the LLM itself decides when to query a retriever, and `4_ReAct_MultiHop_Agentic_RAG.ipynb` shows
the same idea again using LangGraph's prebuilt agent loop instead of a hand-built graph.
